In [14]:
import imaplib
import os
from pathlib import Path

env_path = Path(".env")
if env_path.exists():
    for line in env_path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip())

imap_server = os.getenv("IMAP_SERVER", "imap.gmail.com")
email_address = os.getenv("IMAP_EMAIL")
password = os.getenv("IMAP_PASSWORD", "").replace(" ", "")

if not email_address or not password:
    raise RuntimeError("IMAP_EMAIL and IMAP_PASSWORD must be set in .env")

mail = imaplib.IMAP4_SSL(imap_server)
mail.login(email_address, password)
mail.select("inbox")
print('Connected to mailbox')

Connected to mailbox


---

In [15]:
import imaplib
import email
from email.header import decode_header, make_header
import os
from pathlib import Path
import pdfplumber

env_path = Path(".env")
if env_path.exists():
    for line in env_path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip())

IMAP_SERVER = os.getenv("IMAP_SERVER", "imap.gmail.com")
EMAIL = os.getenv("IMAP_EMAIL")
PASSWORD = os.getenv("IMAP_PASSWORD", "").replace(" ", "")
SAVE_DIR = os.getenv("SAVE_DIR", "./downloads")
if not EMAIL or not PASSWORD:
    raise RuntimeError("IMAP_EMAIL and IMAP_PASSWORD must be set in .env")
os.makedirs(SAVE_DIR, exist_ok=True)

In [16]:
# 1. IMAP 접속
mail = imaplib.IMAP4_SSL(IMAP_SERVER)
mail.login(EMAIL, PASSWORD)
mail.select("inbox")

('OK', [b'4'])

In [17]:
# 2. 메일 검색 (제목 필터는 직접 처리)
result, data = mail.search(None, "ALL")
mail_ids = data[0].split()
target_mail = None
# 최근 메일부터 역순 탐색
for mail_id in reversed(mail_ids):
    result, data = mail.fetch(mail_id, "(RFC822)")
    raw_email = data[0][1]
    msg = email.message_from_bytes(raw_email)
    # 제목 디코딩
    subject = str(make_header(decode_header(msg["Subject"])))
    if "[발주서]" in subject:
        print("대상 메일:", subject)
        target_mail = msg
        break
if target_mail is None:
    print(" 조건에 맞는 메일 없음")
    exit()

대상 메일: [발주서] PO-2026-0428-915


In [18]:
# 3. PDF 첨부파일 다운로드
pdf_files = []
for part in target_mail.walk():
    content_disposition = (part.get("Content-Disposition") or "").lower()
    filename = part.get_filename()
    
    if not filename or "attachment" not in content_disposition:
        continue
    
    filename = str(make_header(decode_header(filename)))
    
    if not filename.lower().endswith(".pdf"):
        continue
    
    filepath = os.path.join(SAVE_DIR, filename)
    with open(filepath, "wb") as f:
        f.write(part.get_payload(decode=True))
    print(f"다운로드 완료: {filepath}")
    pdf_files.append(filepath)

if not pdf_files:
    print("PDF 첨부파일을 찾지 못했습니다.")

다운로드 완료: ./downloads\발주서_PO-2026-0428-915.pdf


In [19]:
# 4. PDF 텍스트 추출
for pdf_path in pdf_files:
    print(f"\n텍스트 추출: {pdf_path}")
    with pdfplumber.open(pdf_path) as pdf:
        full_text = ""
        for page in pdf.pages:
            text = page.extract_text()
            if text:
                full_text += text + "\n"
    print("---- 추출 텍스트 ----")
    print(full_text[:]) # 일부만 출력


텍스트 추출: ./downloads\발주서_PO-2026-0428-915.pdf
---- 추출 텍스트 ----
발 주 서 / 대외발송용
[발주문서 PO-2026-0428-915] 발행일자 2026. 04. 28.
성과관리시스템 운영지원 및 자동화 문서
발 주 서
성과관리시스템 운영지원 및 자동화 고도화
2026. 04.
샘플테크 정보전략실
사업명: 성과관리시스템 운영지원 및 자동화 고도화 발주번호: PO-2026-0428-915
계약유형: 총액계약(부가세 별도) 계약방식: 수의발주(사전 견적 협의)
발주일자: 2026. 04. 28. 착수예정일: 2026. 04. 29.
납품기한: 2026. 05. 19. 회신기한: 2026. 04. 30.
발주기관: 샘플테크 수신처: 메일자동화솔루션 주식회사
납품장소: 샘플테크 정보전략실 / 운영공유폴더 총 계약금액: 5,940,000원
귀사와 협의한 과업 범위 및 공급 조건에 따라 아래와 같이 발주하오니, 첨부 세부내역을 확인하시고 회신기한 내
접수 여부를 알려주시기 바랍니다.
- 1 -
발 주 서 / 대외발송용
1. 사업 개요
추진배경:
1. 성과관리시스템 운영 과정에서 발주서 및 증빙자료를 메일로 수신한 뒤 수작업으로 저장, 정리, 공유하고 있어 업
무부하가 발생하고 있습니다.
2. 첨부 PDF의 저장 위치, 파일명 규칙, 추출 결과 형식이 담당자별로 달라 후속 집계와 보고 자료 작성에 시간이 소
요되고 있습니다.
발주목적:
1. 대외 발송용 발주서 형식을 표준화하여 문서 완성도와 재사용성을 높입니다.
2. 메일로 수신한 발주서 PDF를 자동 저장하고, 구조화된 CSV 데이터로 즉시 전환할 수 있는 흐름을 구축합니다.
발주범위:
1. 정기 유지보수 및 운영지원
2. IMAP 기반 첨부파일 자동 다운로드
3. PDF 텍스트 및 표 데이터 추출
4. CSV 리포트 자동 생성 및 운영 문서 정비
2. 거래처 및 담당자
발주처: 샘플테크 공급사: 메일자동화솔루션 주식회사
부서: 정보전략실 대표자: 김민수
검수부서: 성과관리운

---
# 학습용 코드 구조

프로젝트 파일을 `src/`와 `web/`으로 정리한 뒤의 학습용 섹션입니다.
`src/`에는 Python 백엔드와 PDF 처리 코드가 있고, `web/`에는 브라우저 화면 파일이 있습니다.
HTML/CSS/JS는 깊게 쪼개지 않고, 백엔드 Python 흐름을 함수 단위로 나눠 설명합니다.

## 현재 디렉터리 구조

```text
proj/
  src/                  # Python 백엔드, 메일, PDF, 분석 코드
  web/                  # HTML/CSS/JS 화면 파일
  downloads/            # 메일에서 받은 PDF
  generated_purchase_orders/  # 샘플 PDF 생성 결과
  프로젝트.ipynb        # 실습/학습 노트북
```

## 전체 처리 흐름

1. 브라우저가 `src/mail_ui_server.py` 서버에 접속합니다.
2. `web/flow-ui.js`가 `/api/mail`을 호출하면 `src/inbox_query.py`가 inbox 메일을 조회합니다.
3. 사용자가 메일을 선택하고 다운로드를 누르면 `src/pdf_downloader.py`가 PDF 첨부파일을 `downloads/`에 저장합니다.
4. CSV 변환 버튼을 누르면 `src/csv_converter.py`가 파일 경로를 검증한 뒤 PDF 파서로 넘깁니다.
5. PDF 파서는 `pdfplumber`로 텍스트와 표를 먼저 추출하고, 발주번호/품목/일정 같은 항목을 구조화합니다.
6. 단어 분석 버튼을 누르면 `src/word_analyzer.py`가 단어 빈도를 계산합니다.
7. `src/word_cloud.py`는 단어 빈도에 따라 화면에 표시할 글자 크기와 색상 데이터를 만듭니다.

## src/notebook_workflow.py - 위쪽 실습 코드를 실제 .py로 옮긴 버전

- `# 학습용 코드 구조` 위쪽에서 작성한 절차형 코드를 거의 그대로 함수로 감싼 파일입니다.
- 노트북에서 한 셀씩 실행하던 흐름을 `.py`에서도 import해서 재사용할 수 있습니다.
- 실제 웹 UI는 더 구조화된 `inbox_query.py`, `pdf_downloader.py` 등을 쓰지만, 공부할 때는 이 파일이 위쪽 실습 코드와 가장 직접적으로 대응됩니다.

### import와 환경설정 읽기

- 위쪽 셀의 `.env` 읽기 코드를 `load_notebook_env()` 함수로 옮겼습니다.
- `IMAP_SERVER`, `EMAIL`, `PASSWORD`, `SAVE_DIR` 이름을 그대로 유지해서 기존 셀과 비교하기 쉽습니다.

In [20]:
from __future__ import annotations

import email
import imaplib
import os
import re
from email.header import decode_header, make_header
from pathlib import Path

import pdfplumber


DEFAULT_PURCHASE_MAIL_TERMS = (
    "발주서",
    "발주",
    "구매요청",
    "구매 요청",
    "주문서",
    "purchase order",
    "po-",
    "po_",
    "p/o",
    "견적서",
    "견적",
    "계약서",
    "납품요청",
    "납품 요청",
    "검수요청",
    "검수 요청",
)

def load_notebook_env(env_path: Path = Path(".env")) -> dict[str, str]:
    if env_path.exists():
        for line in env_path.read_text(encoding="utf-8").splitlines():
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, value = line.split("=", 1)
            os.environ.setdefault(key.strip(), value.strip())

    imap_server = os.getenv("IMAP_SERVER", "imap.gmail.com")
    email_address = os.getenv("IMAP_EMAIL")
    password = os.getenv("IMAP_PASSWORD", "").replace(" ", "")
    save_dir = os.getenv("SAVE_DIR", "./downloads")

    if not email_address or not password:
        raise RuntimeError("IMAP_EMAIL and IMAP_PASSWORD must be set in .env")

    return {
        "IMAP_SERVER": imap_server,
        "EMAIL": email_address,
        "PASSWORD": password,
        "SAVE_DIR": save_dir,
    }


### 1. IMAP 접속

- 위쪽 셀의 `mail = imaplib.IMAP4_SSL(...)` 흐름을 `connect_inbox()`로 감쌌습니다.
- 반환된 `mail` 객체는 다음 단계의 검색 함수에서 그대로 사용합니다.

In [21]:
def connect_inbox(imap_server: str, email_address: str, password: str) -> imaplib.IMAP4_SSL:
    # 1. IMAP 접속
    mail = imaplib.IMAP4_SSL(imap_server)
    mail.login(email_address, password)
    mail.select("inbox")
    print("Connected to mailbox")
    return mail


### 2. 메일 검색

- 위쪽 셀처럼 `mail.search(None, "ALL")`로 전체 메일을 찾고, 최신 메일부터 역순으로 확인합니다.
- 제목을 디코딩한 뒤 `[발주서]`가 들어간 첫 메일을 대상 메일로 선택합니다.

In [22]:
def compact_search_text(value: str) -> str:
    return re.sub(r"[^0-9a-z가-힣]+", "", value.lower())

def build_purchase_mail_terms(subject_keywords: str | list[str] | tuple[str, ...] = "[발주서]") -> list[str]:
    if isinstance(subject_keywords, str):
        raw_terms = [term.strip() for term in re.split(r"[,;/\n]+", subject_keywords) if term.strip()]
    else:
        raw_terms = [term.strip() for term in subject_keywords if term.strip()]

    if not raw_terms:
        raw_terms = list(DEFAULT_PURCHASE_MAIL_TERMS)

    normalized_raw_terms = {compact_search_text(term) for term in raw_terms}
    default_triggers = {"발주서", "발주", "purchaseorder", "po"}
    if normalized_raw_terms & default_triggers:
        terms = [*raw_terms, *DEFAULT_PURCHASE_MAIL_TERMS]
    else:
        terms = raw_terms

    deduplicated: list[str] = []
    seen: set[str] = set()
    for term in terms:
        key = compact_search_text(term)
        if not key or key in seen:
            continue
        seen.add(key)
        deduplicated.append(term)
    return deduplicated

def subject_matches_terms(subject: str, terms: list[str]) -> list[str]:
    normalized_subject = subject.lower()
    compact_subject = compact_search_text(subject)
    matched: list[str] = []
    for term in terms:
        normalized_term = term.lower()
        compact_term = compact_search_text(term)
        if normalized_term in normalized_subject or (compact_term and compact_term in compact_subject):
            matched.append(term)
    return matched

def find_latest_purchase_order_mail(
    mail: imaplib.IMAP4_SSL,
    subject_keywords: str | list[str] | tuple[str, ...] = "[발주서]",
) -> email.message.Message | None:
    # 2. 메일 검색 (제목 필터는 직접 처리)
    result, data = mail.search(None, "ALL")
    if result != "OK":
        raise RuntimeError("메일 검색에 실패했습니다.")

    terms = build_purchase_mail_terms(subject_keywords)
    mail_ids = data[0].split()
    target_mail = None
    # 최근 메일부터 역순 탐색
    for mail_id in reversed(mail_ids):
        result, data = mail.fetch(mail_id, "(RFC822)")
        if result != "OK" or not data or not isinstance(data[0], tuple):
            continue

        raw_email = data[0][1]
        msg = email.message_from_bytes(raw_email)
        # 제목 디코딩
        subject = str(make_header(decode_header(msg["Subject"])))
        matched_terms = subject_matches_terms(subject, terms)
        if matched_terms:
            print("대상 메일:", subject)
            print("매칭 키워드:", ", ".join(matched_terms))
            target_mail = msg
            break

    if target_mail is None:
        print("조건에 맞는 메일 없음")
    return target_mail


### 3. PDF 첨부파일 다운로드

- 위쪽 셀의 `target_mail.walk()` 반복문을 그대로 함수화했습니다.
- PDF 첨부파일만 `SAVE_DIR`에 저장하고, 저장된 경로 목록을 반환합니다.

In [23]:
def download_pdf_attachments_from_message(target_mail: email.message.Message, save_dir: str = "./downloads") -> list[str]:
    # 3. PDF 첨부파일 다운로드
    os.makedirs(save_dir, exist_ok=True)

    pdf_files: list[str] = []
    for part in target_mail.walk():
        content_disposition = (part.get("Content-Disposition") or "").lower()
        filename = part.get_filename()

        if not filename or "attachment" not in content_disposition:
            continue

        filename = str(make_header(decode_header(filename)))

        if not filename.lower().endswith(".pdf"):
            continue

        filepath = os.path.join(save_dir, filename)
        with open(filepath, "wb") as f:
            f.write(part.get_payload(decode=True))
        print(f"다운로드 완료: {filepath}")
        pdf_files.append(filepath)

    if not pdf_files:
        print("PDF 첨부파일을 찾지 못했습니다.")

    return pdf_files


### 4. PDF 텍스트 추출

- 질문에서 보여준 `pdfplumber.open()`과 `page.extract_text()` 방식 그대로입니다.
- 각 PDF의 전체 텍스트를 출력하고, 딕셔너리로도 반환해서 다음 분석 단계에서 재사용할 수 있습니다.

In [24]:
def extract_pdf_texts(pdf_files: list[str]) -> dict[str, str]:
    # 4. PDF 텍스트 추출
    extracted: dict[str, str] = {}
    for pdf_path in pdf_files:
        print(f"\n텍스트 추출: {pdf_path}")
        with pdfplumber.open(pdf_path) as pdf:
            full_text = ""
            for page in pdf.pages:
                text = page.extract_text()
                if text:
                    full_text += text + "\n"
        print("---- 추출 텍스트 ----")
        print(full_text[:])
        extracted[pdf_path] = full_text
    return extracted


### 위쪽 1~4단계를 한 번에 실행하는 함수

- 노트북 셀을 하나씩 실행하는 대신 `.py`에서는 `run_notebook_purchase_order_flow()` 하나로 전체 흐름을 실행할 수 있습니다.
- 내부에서는 위에서 만든 네 함수가 순서대로 호출됩니다.

In [25]:
def run_notebook_purchase_order_flow(subject_keywords: str | list[str] | tuple[str, ...] = "[발주서]") -> dict[str, object]:
    settings = load_notebook_env()
    mail = connect_inbox(settings["IMAP_SERVER"], settings["EMAIL"], settings["PASSWORD"])
    try:
        target_mail = find_latest_purchase_order_mail(mail, subject_keywords)
        if target_mail is None:
            return {"mail": None, "pdf_files": [], "texts": {}}

        pdf_files = download_pdf_attachments_from_message(target_mail, settings["SAVE_DIR"])
        texts = extract_pdf_texts(pdf_files)
        return {
            "mail": target_mail,
            "pdf_files": pdf_files,
            "texts": texts,
        }
    finally:
        mail.logout()


### 노트북에서 재사용하는 예시

```python
from src.notebook_workflow import run_notebook_purchase_order_flow

result = run_notebook_purchase_order_flow("[발주서]")
pdf_files = result["pdf_files"]
texts = result["texts"]
```

## src/app_config.py - 공통 설정

- 여러 모듈에서 같이 쓰는 경로와 메일 계정 설정을 모아 둔 파일입니다.
- `src/` 안으로 이동했지만 `BASE_DIR`은 프로젝트 루트를 가리키도록 보정합니다.

### 기본 import, 경로 상수, MailSettings

- 프로젝트 기준 경로와 다운로드/CSV 출력 위치를 정합니다.
- 메일 설정값은 dataclass로 묶어서 전달합니다.

In [26]:
from __future__ import annotations

import os
from dataclasses import dataclass
from email.header import decode_header, make_header
from pathlib import Path


BASE_DIR = Path(__file__).resolve().parent
if BASE_DIR.name == "src":
    BASE_DIR = BASE_DIR.parent
DOWNLOAD_DIR = BASE_DIR / "downloads"
REPORT_OUTPUT = BASE_DIR / "result.csv"


@dataclass(frozen=True)
class MailSettings:
    smtp_server: str
    smtp_port: int
    imap_server: str
    email_address: str
    password: str


NameError: name '__file__' is not defined

### .env 파일 읽기

- `.env` 파일의 `KEY=VALUE` 줄만 읽어서 딕셔너리로 만듭니다.
- 빈 줄과 주석은 건너뜁니다.

In [ ]:
def load_env(path: Path) -> dict[str, str]:
    if not path.exists():
        return {}

    env: dict[str, str] = {}
    for raw_line in path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        env[key.strip()] = value.strip()
    return env

def bootstrap_env() -> None:
    for key, value in load_env(BASE_DIR / ".env").items():
        os.environ.setdefault(key, value)


### 메일 설정 로딩

- 환경변수에서 IMAP/SMTP 설정을 읽습니다.
- 필수 계정이나 비밀번호가 없으면 명확한 오류를 냅니다.

In [ ]:
def load_mail_settings() -> MailSettings:
    bootstrap_env()

    email_address = os.getenv("SMTP_EMAIL") or os.getenv("IMAP_EMAIL")
    password = os.getenv("SMTP_PASSWORD") or os.getenv("IMAP_PASSWORD")
    smtp_server = os.getenv("SMTP_SERVER", "smtp.gmail.com")
    smtp_port = int(os.getenv("SMTP_PORT", "465"))
    imap_server = os.getenv("IMAP_SERVER", "imap.gmail.com")

    if not email_address or not password:
        raise RuntimeError(
            "Missing mail credentials. Set SMTP_EMAIL/SMTP_PASSWORD or IMAP_EMAIL/IMAP_PASSWORD in .env."
        )

    return MailSettings(
        smtp_server=smtp_server,
        smtp_port=smtp_port,
        imap_server=imap_server,
        email_address=email_address,
        password=password,
    )


### MIME 문자열 디코딩

- 메일 제목이나 첨부파일명이 인코딩되어 있을 때 한글로 복원합니다.

In [ ]:
def decode_mime_value(value: str | None) -> str:
    if not value:
        return ""
    return str(make_header(decode_header(value)))


## src/inbox_query.py - inbox 메일 조회

- 메일 목록만 조회하고, 실제 PDF 저장은 아직 하지 않습니다.
- PDF 첨부파일이 있는 메일만 UI 목록에 포함합니다.

### 기본 import와 inbox 접속

- `open_inbox()`는 `.env` 설정으로 IMAP에 로그인하고 inbox를 선택합니다.

In [ ]:
from __future__ import annotations

import email
import imaplib
import re
from datetime import date, timedelta
from email import policy
from email.message import EmailMessage, Message
from email.utils import parsedate_to_datetime

from app_config import decode_mime_value, load_mail_settings


INBOX_MAILBOX = "inbox"
DEFAULT_RELATED_MAIL_TERMS = (
    "발주서",
    "발주",
    "구매요청",
    "구매 요청",
    "주문서",
    "purchase order",
    "po-",
    "po_",
    "p/o",
    "견적서",
    "견적",
    "계약서",
    "납품요청",
    "납품 요청",
    "검수요청",
    "검수 요청",
)

def open_inbox() -> imaplib.IMAP4_SSL:
    settings = load_mail_settings()
    mail = imaplib.IMAP4_SSL(settings.imap_server)
    mail.login(settings.email_address, settings.password)
    result, _ = mail.select(INBOX_MAILBOX)
    if result != "OK":
        mail.logout()
        raise RuntimeError("inbox 메일함을 열 수 없습니다.")
    return mail


### 조회 기간을 IMAP 검색 날짜로 변환

- UI의 최근 7일/30일/전체 값을 IMAP의 `SINCE` 검색 조건으로 바꿉니다.

In [ ]:
def imap_since_date(period: str) -> str | None:
    if period == "all":
        return None
    try:
        days = max(int(period), 1)
    except ValueError:
        days = 7
    since = date.today() - timedelta(days=days - 1)
    return since.strftime("%d-%b-%Y")


### 메일 본문 미리보기 만들기

- 첨부파일은 건너뛰고 `text/plain` 본문만 짧게 모읍니다.
- UI 오른쪽 상세 패널에 보여주는 본문 미리보기입니다.

In [ ]:
def decode_text_part(part: Message | EmailMessage) -> str:
    try:
        content = part.get_content()
        return content if isinstance(content, str) else ""
    except Exception:
        payload = part.get_payload(decode=True)
        if not payload:
            return ""
        charset = part.get_content_charset() or "utf-8"
        return payload.decode(charset, errors="replace")

def body_preview(message: Message | EmailMessage, max_length: int = 260) -> str:
    candidates: list[str] = []
    if message.is_multipart():
        for part in message.walk():
            if part.is_multipart():
                continue
            if part.get_content_disposition() == "attachment":
                continue
            if part.get_content_type() == "text/plain":
                candidates.append(decode_text_part(part))
    elif message.get_content_type() == "text/plain":
        candidates.append(decode_text_part(message))

    text = " ".join(" ".join(candidate.split()) for candidate in candidates if candidate)
    return text[:max_length]


### 첨부파일 정보와 날짜 정리

- 첨부파일 이름, 크기, PDF 여부만 메타데이터로 정리합니다.
- 메일 날짜는 `YYYY-MM-DD HH:MM` 형식으로 바꿉니다.

In [ ]:
def byte_size_label(size: int) -> str:
    if size >= 1024 * 1024:
        return f"{size / (1024 * 1024):.1f} MB"
    if size >= 1024:
        return f"{size / 1024:.0f} KB"
    return f"{size} B"

def list_attachments(message: Message | EmailMessage) -> list[dict[str, object]]:
    attachments: list[dict[str, object]] = []
    for part in message.walk() if message.is_multipart() else [message]:
        filename = part.get_filename()
        if not filename:
            continue

        filename = decode_mime_value(filename)
        payload = part.get_payload(decode=True) or b""
        content_type = part.get_content_type()
        is_pdf = filename.lower().endswith(".pdf") or content_type == "application/pdf"
        attachments.append(
            {
                "name": filename,
                "size": len(payload),
                "sizeLabel": byte_size_label(len(payload)),
                "contentType": content_type,
                "isPdf": is_pdf,
            }
        )
    return attachments

def formatted_mail_date(value: str | None) -> str:
    if not value:
        return ""
    try:
        parsed = parsedate_to_datetime(value)
        if parsed.tzinfo is not None:
            parsed = parsed.astimezone()
        return parsed.strftime("%Y-%m-%d %H:%M")
    except Exception:
        return value


### 관련 키워드 검색 방식

- 이제 `[발주서]` 한 단어만 비교하지 않습니다.
- 기본 키워드에 `발주`, `구매요청`, `purchase order`, `PO`, `견적`, `계약`, `납품요청`, `검수요청` 같은 관련어를 묶어 둡니다.
- 메일 제목, 본문 미리보기, 첨부파일명 중 하나라도 관련 키워드와 맞으면 후보 메일로 가져옵니다.

### UID 검색과 UI용 메일 목록 만들기

- IMAP UID를 기준으로 메일을 가져옵니다.
- 검색어와 PDF 첨부 여부를 검사해 UI 목록을 만듭니다.

In [ ]:
def fetch_message_by_uid(mail: imaplib.IMAP4_SSL, uid: str) -> Message | EmailMessage | None:
    if not uid.isdigit():
        raise RuntimeError(f"잘못된 메일 UID입니다: {uid}")

    result, data = mail.uid("fetch", uid, "(RFC822)")
    if result != "OK":
        return None

    for item in data:
        if isinstance(item, tuple) and item[1]:
            return email.message_from_bytes(item[1], policy=policy.default)
    return None

def search_mail_uids(mail: imaplib.IMAP4_SSL, period: str) -> list[str]:
    since = imap_since_date(period)
    if since:
        result, data = mail.uid("search", None, "SINCE", since)
    else:
        result, data = mail.uid("search", None, "ALL")

    if result != "OK":
        raise RuntimeError("메일 검색에 실패했습니다.")
    return [uid.decode("ascii") for uid in data[0].split()]

def compact_search_text(value: str) -> str:
    return re.sub(r"[^0-9a-z가-힣]+", "", value.lower())

def split_search_terms(search_text: str) -> list[str]:
    raw_terms = [term.strip() for term in re.split(r"[,;/\n]+", search_text) if term.strip()]
    if not raw_terms:
        return list(DEFAULT_RELATED_MAIL_TERMS)

    normalized_raw_terms = {compact_search_text(term) for term in raw_terms}
    default_triggers = {"발주서", "발주", "purchaseorder", "po"}
    if normalized_raw_terms & default_triggers:
        terms = [*raw_terms, *DEFAULT_RELATED_MAIL_TERMS]
    else:
        terms = raw_terms

    deduplicated: list[str] = []
    seen: set[str] = set()
    for term in terms:
        key = compact_search_text(term)
        if not key or key in seen:
            continue
        seen.add(key)
        deduplicated.append(term)
    return deduplicated

def matched_search_terms(haystack: str, terms: list[str]) -> list[str]:
    normalized_haystack = haystack.lower()
    compact_haystack = compact_search_text(haystack)
    matched: list[str] = []
    for term in terms:
        normalized_term = term.lower()
        compact_term = compact_search_text(term)
        if normalized_term in normalized_haystack or (compact_term and compact_term in compact_haystack):
            matched.append(term)
    return matched

def message_to_summary(uid: str, message: Message | EmailMessage) -> dict[str, object]:
    subject = decode_mime_value(str(message.get("Subject", "")))
    from_value = decode_mime_value(str(message.get("From", "")))
    preview = body_preview(message)
    attachments = list_attachments(message)
    return {
        "id": uid,
        "receivedAt": formatted_mail_date(str(message.get("Date", ""))),
        "from": from_value,
        "subject": subject,
        "body": preview,
        "attachments": attachments,
    }

def list_inbox_mails(search_text: str, period: str, limit: int = 50) -> list[dict[str, object]]:
    search_terms = split_search_terms(search_text)
    mails: list[dict[str, object]] = []
    mail = open_inbox()
    try:
        for uid in reversed(search_mail_uids(mail, period)):
            message = fetch_message_by_uid(mail, uid)
            if message is None:
                continue

            summary = message_to_summary(uid, message)
            attachment_names = " ".join(str(item["name"]) for item in summary["attachments"])
            haystack = " ".join(
                [
                    str(summary["subject"]),
                    str(summary["from"]),
                    str(summary["body"]),
                    attachment_names,
                ]
            ).lower()
            matched_terms = matched_search_terms(haystack, search_terms)
            if search_terms and not matched_terms:
                continue

            if not any(attachment["isPdf"] for attachment in summary["attachments"]):
                continue

            summary["matchedTerms"] = matched_terms
            mails.append(summary)
            if len(mails) >= limit:
                break
    finally:
        mail.logout()

    return mails


## src/pdf_downloader.py - PDF 첨부 다운로드

- 사용자가 다운로드 버튼을 눌렀을 때만 실제 PDF 파일을 저장합니다.
- 저장 결과는 CSV 변환과 단어 분석 대상이 됩니다.

### 파일명 정리와 중복 파일 처리

- 위험한 파일명 문자를 `_`로 바꿉니다.
- 같은 이름이 이미 있으면 `_2`, `_3`처럼 번호를 붙입니다.

In [ ]:
from __future__ import annotations

import re
from pathlib import Path

from app_config import BASE_DIR, DOWNLOAD_DIR, decode_mime_value
from inbox_query import byte_size_label, fetch_message_by_uid, open_inbox

def safe_filename(filename: str) -> str:
    name = Path(filename).name.strip()
    name = re.sub(r'[<>:"/\\|?*\x00-\x1f]', "_", name)
    return name or "attachment.pdf"

def unique_path(directory: Path, filename: str) -> Path:
    candidate = directory / filename
    if not candidate.exists():
        return candidate

    stem = candidate.stem
    suffix = candidate.suffix
    index = 2
    while True:
        next_candidate = directory / f"{stem}_{index}{suffix}"
        if not next_candidate.exists():
            return next_candidate
        index += 1


### PDF 첨부파일 다운로드

- 선택한 메일 UID만 다시 열어서 PDF 첨부파일을 저장합니다.
- `downloads/` 안에 저장하고 UI가 쓸 수 있는 상대 경로를 반환합니다.

In [ ]:
def download_pdf_attachments(mail_ids: list[str]) -> list[dict[str, object]]:
    if not mail_ids:
        return []

    DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
    downloaded: list[dict[str, object]] = []
    mail = open_inbox()
    try:
        for uid in mail_ids:
            message = fetch_message_by_uid(mail, uid)
            if message is None:
                continue

            subject = decode_mime_value(str(message.get("Subject", "")))
            for part in message.walk() if message.is_multipart() else [message]:
                filename = part.get_filename()
                if not filename:
                    continue

                filename = safe_filename(decode_mime_value(filename))
                content_type = part.get_content_type()
                if not (filename.lower().endswith(".pdf") or content_type == "application/pdf"):
                    continue

                payload = part.get_payload(decode=True)
                if payload is None:
                    continue

                destination = unique_path(DOWNLOAD_DIR, filename)
                destination.write_bytes(payload)
                relative_path = destination.relative_to(BASE_DIR).as_posix()
                downloaded.append(
                    {
                        "id": relative_path,
                        "name": destination.name,
                        "relativePath": relative_path,
                        "size": len(payload),
                        "sizeLabel": byte_size_label(len(payload)),
                        "sourceMailId": uid,
                        "sourceSubject": subject,
                    }
                )
    finally:
        mail.logout()

    return downloaded


## src/csv_converter.py - CSV 변환 진입점

- UI가 넘긴 PDF 경로를 검증하고, 실제 CSV 변환 함수로 연결합니다.
- 실제 PDF 텍스트/표 추출은 `src/send_purchase_orders.py`의 파서를 재사용합니다.

### 다운로드 PDF 경로 검증

- 사용자가 넘긴 파일이 `downloads/` 폴더 안에 있는지 확인합니다.
- PDF가 아니거나 파일이 없으면 변환 전에 오류를 냅니다.

In [ ]:
from __future__ import annotations

from pathlib import Path

from app_config import BASE_DIR, DOWNLOAD_DIR, REPORT_OUTPUT
from send_purchase_orders import export_purchase_orders_to_csv_report

def resolve_downloaded_pdf(relative_path: str) -> Path:
    candidate = (BASE_DIR / relative_path).resolve()
    download_root = DOWNLOAD_DIR.resolve()
    if not candidate.is_relative_to(download_root):
        raise RuntimeError(f"다운로드 폴더 밖의 파일은 변환할 수 없습니다: {relative_path}")
    if candidate.suffix.lower() != ".pdf":
        raise RuntimeError(f"PDF 파일만 변환할 수 있습니다: {relative_path}")
    if not candidate.exists():
        raise RuntimeError(f"파일을 찾을 수 없습니다: {relative_path}")
    return candidate


### CSV 변환 함수 호출

- 검증된 PDF 경로 목록을 `export_purchase_orders_to_csv_report()`에 넘깁니다.
- 결과 CSV 경로를 UI로 반환합니다.

In [ ]:
def convert_pdfs_to_csv(files: list[str]) -> dict[str, object]:
    if not files:
        raise RuntimeError("CSV로 변환할 PDF를 선택하세요.")

    pdf_paths = [resolve_downloaded_pdf(file_path) for file_path in files]
    report_path = export_purchase_orders_to_csv_report(pdf_paths, REPORT_OUTPUT)
    return {
        "convertedFiles": files,
        "reportPath": report_path.relative_to(BASE_DIR).as_posix(),
    }


## src/send_purchase_orders.py 중 재사용하는 PDF 파서

- 전체 PDF 생성기가 아니라, CSV 변환과 단어 분석에서 재사용하는 핵심 파서 부분만 봅니다.
- CSV 변환은 먼저 `pdfplumber`로 텍스트와 표를 추출합니다.

### PDF에서 텍스트와 표 추출

- `page.extract_text()`로 텍스트를 뽑고, `page.extract_tables()`로 표도 함께 가져옵니다.

In [ ]:
def extract_pdf_assets(pdf_path: Path) -> dict[str, Any]:
    texts: list[str] = []
    tables: list[list[list[str | None]]] = []
    with pdfplumber.open(str(pdf_path)) as pdf:
        page_count = len(pdf.pages)
        for page in pdf.pages:
            text = page.extract_text() or ""
            if text:
                texts.append(text)
            for table in page.extract_tables():
                tables.append(table)
    return {
        "text": "\n".join(texts),
        "tables": tables,
        "page_count": page_count,
    }


### PDF 한 건을 발주서 데이터로 파싱

- 텍스트 줄과 표 데이터를 함께 사용해서 기본 정보, 품목, 산출물, 추진 일정을 구조화합니다.

In [ ]:
def parse_purchase_order_document(pdf_path: Path) -> dict[str, Any]:
    assets = extract_pdf_assets(pdf_path)
    tables = assets["tables"]
    text = assets["text"]
    lines = normalized_lines(text)
    fields = extract_labeled_fields(tables)

    buyer_contact, buyer_phone = split_contact_pair(nth_field(fields, "담당자", 0))
    supplier_contact, supplier_phone = split_contact_pair(nth_field(fields, "담당자", 1))
    buyer_email = nth_field(fields, "이메일", 0)
    supplier_email = nth_field(fields, "이메일", 1)

    summary = {
        "pdf_file": pdf_path.name,
        "pdf_path": str(pdf_path.resolve()),
        "source_type": "downloaded_from_mail" if pdf_path.resolve().is_relative_to(DOWNLOAD_DIR.resolve()) else "generated_locally",
        "page_count": assets["page_count"],
        "document_title": DOCUMENT_TITLE,
        "document_class": DOCUMENT_CLASS,
        "issuer": DOCUMENT_ISSUER,
        "project_name": PROJECT_NAME,
        "order_number": first_field(fields, "발주번호") or extract_labeled_value(lines, "발주번호"),
        "order_date": normalize_date(first_field(fields, "발주일자") or extract_labeled_value(lines, "발주일자")),
        "planned_start_date": normalize_date(first_field(fields, "착수예정일")),
        "delivery_date": normalize_date(first_field(fields, "납품기한") or extract_labeled_value(lines, "납품기한")),
        "response_due_date": normalize_date(first_field(fields, "회신기한")),
        "contract_type": first_field(fields, "계약유형", CONTRACT_TYPE),
        "procurement_method": first_field(fields, "계약방식", PROCUREMENT_METHOD),
        "delivery_location": first_field(fields, "납품장소", DELIVERY_LOCATION),
        "buyer_company": first_field(fields, "발주기관") or first_field(fields, "발주처") or BUYER["기관명"],
        "buyer_department": first_field(fields, "부서") or first_field(fields, "담당부서") or BUYER["부서"],
        "inspection_department": first_field(fields, "검수부서", BUYER["검수부서"]),
        "buyer_contact": buyer_contact,
        "buyer_phone": buyer_phone,
        "buyer_email": buyer_email,
        "buyer_address": first_field(fields, "주소", BUYER["주소"]),
        "supplier_company": first_field(fields, "수신처") or first_field(fields, "공급사") or SUPPLIER["회사명"],
        "supplier_ceo": first_field(fields, "대표자", SUPPLIER["대표자"]),
        "supplier_registration_number": first_field(fields, "사업자등록번호", SUPPLIER["사업자등록번호"]),
        "supplier_contact": supplier_contact,
        "supplier_phone": supplier_phone,
        "supplier_email": supplier_email,
        "supplier_address": nth_field(fields, "주소", 1, SUPPLIER["주소"]),
        "background": lines_to_summary(BACKGROUND_POINTS),
        "purpose": lines_to_summary(OBJECTIVE_POINTS),
        "scope": lines_to_summary(SCOPE_POINTS),
        "inspection_terms": INSPECTION_TERMS,
        "payment_terms": PAYMENT_TERMS,
        "quality_requirements": lines_to_summary(QUALITY_REQUIREMENTS),
        "security_requirements": lines_to_summary(SECURITY_REQUIREMENTS),
        "support_requirements": lines_to_summary(SUPPORT_REQUIREMENTS),
        "reporting_requirements": REPORTING_TERMS,
        "reply_method": MAIL_REPLY_METHOD,
        "e_document_terms": E_DOCUMENT_TERMS,
        "special_notes_summary": first_field(fields, "특기사항요약", lines_to_summary(SPECIAL_NOTES)),
        "supply_amount": parse_amount(first_field(fields, "공급가액")),
        "vat": parse_amount(first_field(fields, "부가가치세")),
        "total_amount": parse_amount(first_field(fields, "총 계약금액")),
    }

    items = extract_line_items_from_fields(fields) or extract_line_items_from_tables(tables)
    deliverables = extract_deliverables_from_fields(fields) or extract_deliverables_from_tables(tables)
    milestones = extract_milestones_from_fields(fields) or extract_milestones_from_tables(tables)

    summary["item_count"] = len(items)
    summary["deliverable_count"] = len(deliverables)
    summary["milestone_count"] = len(milestones)
    summary["item_summary"] = " | ".join(f'{item["item_code"]}:{item["item_name"]}' for item in items)
    summary["deliverable_summary"] = " | ".join(f'{deliverable["deliverable_id"]}:{deliverable["deliverable_name"]}' for deliverable in deliverables)
    summary["milestone_summary"] = " | ".join(
        f'{milestone["phase"]}({milestone["start_date"]}~{milestone["end_date"]})' for milestone in milestones
    )

    item_rows = [{**summary, **item} for item in items]
    deliverable_rows = [
        {
            "pdf_file": summary["pdf_file"],
            "pdf_path": summary["pdf_path"],
            "source_type": summary["source_type"],
            "order_number": summary["order_number"],
            "project_name": summary["project_name"],
            **deliverable,
        }
        for deliverable in deliverables
    ]
    milestone_rows = [
        {
            "pdf_file": summary["pdf_file"],
            "pdf_path": summary["pdf_path"],
            "source_type": summary["source_type"],
            "order_number": summary["order_number"],
            "project_name": summary["project_name"],
            **milestone,
        }
        for milestone in milestones
    ]

    return {
        "summary": summary,
        "items": item_rows,
        "deliverables": deliverable_rows,
        "milestones": milestone_rows,
    }


### 여러 PDF를 하나의 CSV 리포트로 저장

- 선택한 PDF들을 하나씩 파싱한 뒤, 문서별 섹션과 행을 만들어 CSV로 씁니다.

In [ ]:
def export_purchase_orders_to_csv_report(pdf_paths: list[Path], report_path: Path) -> Path:
    documents = [parse_purchase_order_document(pdf_path) for pdf_path in pdf_paths]
    if not documents:
        raise RuntimeError("No purchase order PDFs were selected for report export.")

    writable_report_path = resolve_writable_report_path(report_path)
    cleanup_legacy_report_outputs(writable_report_path)

    rows: list[list[Any]] = []
    rows.append(["발주서 통합 리포트", "", ""])
    rows.append([f"생성기준일 {display_date(ORDER_DATE)} / 문서건수 {len(documents)}", "", ""])
    rows.append(["", "", ""])

    for doc_index, document in enumerate(documents, start=1):
        summary = document["summary"]
        items = document["items"]
        deliverables = document["deliverables"]
        milestones = document["milestones"]

        rows.append([f"[문서 {doc_index}]", summary["pdf_file"], ""])
        rows.append(["", "", ""])

        append_section_title(rows, "문서 기본 정보")
        for group, field, value in (
            ("기본", "발주번호", summary["order_number"]),
            ("기본", "사업명", summary["project_name"]),
            ("기본", "문서구분", summary["document_class"]),
            ("기본", "발행부서", summary["issuer"]),
            ("기본", "PDF 경로", summary["pdf_path"]),
            ("기본", "문서 원천", summary["source_type"]),
            ("기본", "페이지 수", summary["page_count"]),
            ("일정", "발주일자", summary["order_date"]),
            ("일정", "착수예정일", summary["planned_start_date"]),
            ("일정", "납품기한", summary["delivery_date"]),
            ("일정", "회신기한", summary["response_due_date"]),
            ("계약", "계약유형", summary["contract_type"]),
            ("계약", "계약방식", summary["procurement_method"]),
            ("계약", "납품장소", summary["delivery_location"]),
        ):
            append_report_row(rows, group, field, value)
        rows.append(["", "", ""])

        append_section_title(rows, "금액 요약")
        for group, field, value in (
            ("금액", "공급가액", summary["supply_amount"]),
            ("금액", "부가세", summary["vat"]),
            ("금액", "총 계약금액", summary["total_amount"]),
            ("건수", "품목 수", summary["item_count"]),
            ("건수", "산출물 수", summary["deliverable_count"]),
            ("건수", "일정 수", summary["milestone_count"]),
        ):
            append_report_row(rows, group, field, value)
        rows.append(["", "", ""])

        append_section_title(rows, "거래처 정보")
        for group, field, value in (
            ("발주처", "기관명", summary["buyer_company"]),
            ("발주처", "부서", summary["buyer_department"]),
            ("발주처", "검수부서", summary["inspection_department"]),
            ("발주처", "담당자", summary["buyer_contact"]),
            ("발주처", "연락처", summary["buyer_phone"]),
            ("발주처", "이메일", summary["buyer_email"]),
            ("발주처", "주소", summary["buyer_address"]),
            ("공급사", "회사명", summary["supplier_company"]),
            ("공급사", "대표자", summary["supplier_ceo"]),
            ("공급사", "사업자등록번호", summary["supplier_registration_number"]),
            ("공급사", "담당자", summary["supplier_contact"]),
            ("공급사", "연락처", summary["supplier_phone"]),
            ("공급사", "이메일", summary["supplier_email"]),
            ("공급사", "주소", summary["supplier_address"]),
        ):
            append_report_row(rows, group, field, value)
        rows.append(["", "", ""])

        append_section_title(rows, "업무 요건")
        for group, field, value in (
            ("요건", "추진배경", summary["background"]),
            ("요건", "발주목적", summary["purpose"]),
            ("요건", "발주범위", summary["scope"]),
            ("조건", "검수조건", summary["inspection_terms"]),
            ("조건", "결제조건", summary["payment_terms"]),
            ("조건", "품질요구", summary["quality_requirements"]),
            ("조건", "보안요구", summary["security_requirements"]),
            ("조건", "지원요구", summary["support_requirements"]),
            ("운영", "보고체계", summary["reporting_requirements"]),
            ("운영", "회신방법", summary["reply_method"]),
            ("운영", "전자문서", summary["e_document_terms"]),
            ("운영", "특기사항", summary["special_notes_summary"]),
        ):
            append_report_row(rows, group, field, value)
        rows.append(["", "", ""])

        append_section_title(rows, "품목 요약")
        for item in items:
            append_report_row(
                rows,
                item["item_code"],
                item["item_category"],
                f'{item["item_name"]} / 수량 {item["quantity"]} / 단가 {won(int(item["unit_price"]))} / 금액 {won(int(item["amount"]))} / 산출물 {item["deliverable_ref"]}',
            )
        rows.append(["", "", ""])

        append_section_title(rows, "품목 상세")
        for item in items:
            for field, value in (
                ("구분", item["item_category"]),
                ("품목명", item["item_name"]),
                ("규격", item["specification"]),
                ("단위", item["unit"]),
                ("수량", item["quantity"]),
                ("단가", won(int(item["unit_price"]))),
                ("금액", won(int(item["amount"]))),
                ("연계 산출물", item["deliverable_ref"]),
                ("비고", item["item_note"]),
            ):
                append_report_row(rows, item["item_code"], field, value)
            rows.append(["", "", ""])

        append_section_title(rows, "산출물 계획")
        for deliverable in deliverables:
            for field, value in (
                ("산출물명", deliverable["deliverable_name"]),
                ("형식", deliverable["deliverable_format"]),
                ("제출기한", deliverable["due_date"]),
                ("검수기준", deliverable["acceptance_criteria"]),
                ("담당", deliverable["owner"]),
            ):
                append_report_row(rows, deliverable["deliverable_id"], field, value)
            rows.append(["", "", ""])

        append_section_title(rows, "추진 일정")
        for milestone in milestones:
            for field, value in (
                ("시작일", milestone["start_date"]),
                ("종료일", milestone["end_date"]),
                ("주요 작업", milestone["major_tasks"]),
                ("검토주체", milestone["review_owner"]),
            ):
                append_report_row(rows, milestone["phase"], field, value)
            rows.append(["", "", ""])

        rows.append(["", "", ""])
        rows.append(["", "", ""])

    writable_report_path.parent.mkdir(parents=True, exist_ok=True)
    with writable_report_path.open("w", newline="", encoding="utf-8-sig") as handle:
        writer = csv.writer(handle)
        writer.writerows(rows)
    print(f"report csv exported: {writable_report_path}")
    return writable_report_path


## src/word_analyzer.py - 단어 등장횟수 분석

- PDF 텍스트를 추출한 뒤 단어 단위로 분석합니다.
- 불용어를 조정하면 워드클라우드와 빈도표 결과가 더 좋아집니다.

### 불용어 목록

- 모든 문서에 반복되지만 분석 의미가 약한 단어를 제외합니다.

In [ ]:
from __future__ import annotations

import re
from collections import Counter

from csv_converter import resolve_downloaded_pdf
from send_purchase_orders import extract_pdf_assets

STOPWORDS = {
    # General particles / connective words
    "그리고",
    "또는",
    "대한",
    "기준",
    "관련",
    "아래",
    "위와",
    "같이",
    "통해",
    "경우",
    "이내",
    "발주서",
    "발주",
    "문서",
    "확인",
    "제출",
    "포함",
    "합니다",
    "있습니다",
    "수",
    "및",
    "등",
    "하는",
    "하여",
    "하며",
    "하고",
    "해야",
    "한다",
    "있다",
    "있는",
    "있어야",
    "따라",
    "위해",
    "통한",
    "대해",
    "각",
    "본",
    "해당",
    "전반",
    "일부",
    "전체",
    "내역",
    "사항",
    "한다는",
    # RFP / document boilerplate
    "rfp",
    "더미",
    "제안",
    "제안서",
    "제안요청",
    "제안요청서",
    "제안사가",
    "제안사는",
    "제시",
    "제시하여야",
    "작성",
    "작성하여야",
    "제출물",
    "산출물",
    "산출정보",
    "요구",
    "요구사항",
    "요구사항명",
    "항목",
    "구분",
    "내용",
    "설명",
    "기준서",
    "명세서",
    "기능명세서",
    "테스트결과서",
    "보고서",
    "매뉴얼",
    "문서는",
    "문서화",
    "키워드",
    "id",
    "sfr",
    "dar",
    "ser",
    "qur",
    "pmr",
    "psr",
    # Generic project / management terms that dominated all five RFPs
    "사업",
    "시스템",
    "관리",
    "운영",
    "기능",
    "방안",
    "방안을",
    "주요",
    "단계",
    "절차",
    "절차를",
    "범위",
    "기반",
    "수행",
    "처리",
    "지원",
    "서비스",
    "담당자",
    "기관",
    "업무",
    "계약",
    "기간",
    "일정",
    "현황",
    "조회",
    "입력",
    "결과",
    "검증",
    "테스트",
    "품질",
    "품질관리",
    "개선",
    "변경",
    "설계",
    "정책",
    "이력",
    "로그",
    "오류",
    "누락",
    "실패",
    "지연",
    "리스크",
    "위험",
    "대응",
    "조치",
    "안정화",
    "보관",
    "가상",
    "it",
    # Second pass after reviewing the five dummy RFP PDFs
    "분석",
    "데이터",
    "사용자",
    "보안",
    "장애",
    "관리자",
    "개인정보",
    "점검",
    "승인",
    "권한",
    "문의",
    "이슈",
    "대시보드",
    "보호",
    "마스킹",
    "생성",
    "내부",
    "연계",
    "샘플",
    "공공",
    "세부",
    "미흡",
    "목적의",
    "지정",
    "전략",
    "전환",
    "현행",
    "보안관리",
    "성능",
    "종료",
    "가능",
    "기술능력평가",
    "접근",
    "대상",
    "결과를",
    "파일",
    "다운로드",
    "포함하여야",
    "인력",
    "양식",
    "결과서",
    "정보",
    "추진",
    "관리할",
    "기능을",
    "부서",
    "기준을",
    "개발",
    "표준화",
    "외부",
    "중복",
    "착수",
    "저하",
    "이해도",
    "가격평가",
    "월간",
    "일정관리",
    "응답",
    "보고",
    "계정",
    "발주기관",
    "개요",
    "자동",
    "유형",
    "검수",
    "기준에",
    "불일치",
    "필요",
    "실제",
    "목적",
    "알림",
    "상태",
    "제공하여야",
    "화면설계서",
    "접속",
    "검색",
    "재발",
    "관리하여야",
    # RFP phrasing / evaluation boilerplate left after the second pass
    "기대효과",
    "포함하여",
    "구축",
    "기존",
    "교육자료",
    "권한관리",
    "접근성",
    "별도",
    "가능한",
    "표준",
    "기준으로",
    "여부",
    "증가",
    "정기",
    "위한",
    "역량",
    "사업수행계획서",
    "제한",
    "수정",
    "요구사항별",
    "배점",
    "방안은",
    "등록",
    "로그관리",
    "문제",
    "행위",
    "안정화보고서",
    "반복",
    "의사결정",
    "부족",
    "제출하여야",
    "체계를",
    "사업의",
    "합계",
    "제외",
    "이력을",
    "여부를",
    "현황을",
    "보안관리계획서",
    "형식",
    "있도록",
    "수행하여야",
    "명확히",
    "단계별",
    "투입",
    "적용하여야",
    "샘플입니다",
    "제안요청서입니다",
    "시스템은",
    "the",
    "and",
    "for",
    "with",
}


### 텍스트를 단어로 쪼개기

- 한글/영문/숫자 토큰을 찾고, 영문은 소문자로 통일합니다.
- 짧은 단어, 숫자만 있는 토큰, 불용어는 제외합니다.

In [ ]:
def tokenize_text(text: str) -> list[str]:
    tokens: list[str] = []
    for raw_token in re.findall(r"[가-힣A-Za-z0-9]+", text):
        token = raw_token.lower() if raw_token.isascii() else raw_token
        if len(token) < 2 or token in STOPWORDS:
            continue
        if token.isdigit():
            continue
        tokens.append(token)
    return tokens


### PDF 단어 빈도 계산

- 선택한 PDF마다 텍스트를 추출하고 Counter로 빈도를 합산합니다.
- 각 단어에 `count`와 워드클라우드용 `weight`를 붙입니다.

In [ ]:
def analyze_word_counts(files: list[str], top_n: int = 80) -> dict[str, object]:
    if not files:
        raise RuntimeError("분석할 PDF를 선택하세요.")

    pdf_paths = [resolve_downloaded_pdf(file_path) for file_path in files]
    counter: Counter[str] = Counter()
    for pdf_path in pdf_paths:
        assets = extract_pdf_assets(pdf_path)
        counter.update(tokenize_text(str(assets.get("text", ""))))

    total_words = sum(counter.values())
    max_count = max(counter.values(), default=1)
    top_words = [
        {
            "word": word,
            "count": count,
            "weight": round(count / max_count, 4),
        }
        for word, count in counter.most_common(top_n)
    ]
    return {
        "fileCount": len(pdf_paths),
        "totalWords": total_words,
        "uniqueWords": len(counter),
        "topWords": top_words,
    }


## src/word_cloud.py - 워드클라우드 데이터

- 이미지 파일을 만드는 것이 아니라, 브라우저가 그릴 수 있는 표시 정보를 만듭니다.
- 단어 빈도 가중치를 글자 크기와 색상으로 바꿉니다.

### 실제 WordCloud 이미지와 표시값 만들기

- `WordCloud(background_color='white', max_words=2000, font_path=..., random_state=42)` 방식으로 실제 PNG 이미지를 만듭니다.
- UI 호환을 위해 기존 글자 크기/색상 데이터도 함께 유지합니다.


In [ ]:
from __future__ import annotations

import base64
from io import BytesIO
from pathlib import Path

import numpy as np
from PIL import Image
from wordcloud import WordCloud


CLOUD_COLORS = ["#2563a8", "#0f766e", "#c65f19", "#b4233c", "#5b55a3"]
FONT_PATH = Path(r"C:\Windows\Fonts\malgun.ttf")


def build_word_cloud(top_words: list[dict[str, object]], max_words: int = 35) -> list[dict[str, object]]:
    cloud: list[dict[str, object]] = []
    for index, item in enumerate(top_words[:max_words]):
        weight = float(item.get("weight") or 0)
        cloud.append(
            {
                "word": item.get("word", ""),
                "count": item.get("count", 0),
                "weight": weight,
                "sizeRem": round(0.9 + weight * 1.65, 2),
                "color": CLOUD_COLORS[index % len(CLOUD_COLORS)],
            }
        )
    return cloud


def words_to_text(top_words: list[dict[str, object]]) -> str:
    words: list[str] = []
    for item in top_words:
        word = str(item.get("word", "")).strip()
        count = int(item.get("count") or 0)
        if not word or count <= 0:
            continue
        words.extend([word] * count)
    return " ".join(words)


def load_mask(mask_path: str | None = None) -> np.ndarray | None:
    if not mask_path:
        return None

    path = Path(mask_path)
    if not path.exists():
        return None
    return np.array(Image.open(path))


def build_word_cloud_image(
    top_words: list[dict[str, object]],
    mask_path: str | None = None,
    max_words: int = 2000,
) -> str:
    text = words_to_text(top_words)
    if not text:
        return ""

    mask_ar = load_mask(mask_path)
    font_path = str(FONT_PATH) if FONT_PATH.exists() else None
    wordcloud = WordCloud(
        background_color="white",
        max_words=max_words,
        font_path=font_path,
        mask=mask_ar,
        random_state=42,
        width=1200,
        height=800,
        colormap="tab10",
    )
    wordcloud.generate(text)

    image = wordcloud.to_image()
    buffer = BytesIO()
    image.save(buffer, format="PNG")
    encoded = base64.b64encode(buffer.getvalue()).decode("ascii")
    return f"data:image/png;base64,{encoded}"


## src/mail_ui_server.py - 웹 UI/API 서버

- 서버 파일은 기능을 직접 구현하지 않고 각 모듈을 호출하는 라우터 역할만 합니다.
- `web/` 폴더의 HTML/CSS/JS를 정적 파일로 제공합니다.

### 서버 import, 정적 파일 경로, 공통 응답 함수

- HTML/CSS/JS 파일을 제공하고, JSON 응답을 공통 형식으로 보냅니다.

In [ ]:
from __future__ import annotations

import json
import mimetypes
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path
from urllib.parse import parse_qs, urlparse

from app_config import BASE_DIR, load_mail_settings
from csv_converter import convert_pdfs_to_csv
from inbox_query import list_inbox_mails
from pdf_downloader import download_pdf_attachments
from word_analyzer import analyze_word_counts
from word_cloud import build_word_cloud


HOST = "127.0.0.1"
PORT = 8000
WEB_DIR = BASE_DIR / "web"
STATIC_FILES = {
    "/": WEB_DIR / "flow-ui.html",
    "/flow-ui.html": WEB_DIR / "flow-ui.html",
    "/flow-ui.css": WEB_DIR / "flow-ui.css",
    "/flow-ui.js": WEB_DIR / "flow-ui.js",
}


def json_response(handler: BaseHTTPRequestHandler, status: int, payload: dict) -> None:
    body = json.dumps(payload, ensure_ascii=False).encode("utf-8")
    handler.send_response(status)
    handler.send_header("Content-Type", "application/json; charset=utf-8")
    handler.send_header("Content-Length", str(len(body)))
    handler.end_headers()
    handler.wfile.write(body)


def static_response(handler: BaseHTTPRequestHandler, file_path: Path) -> None:
    if not file_path.exists():
        handler.send_error(404)
        return

    body = file_path.read_bytes()
    content_type = mimetypes.guess_type(file_path.name)[0] or "application/octet-stream"
    if file_path.suffix in {".html", ".css", ".js"}:
        content_type = f"{content_type}; charset=utf-8"

    handler.send_response(200)
    handler.send_header("Content-Type", content_type)
    handler.send_header("Content-Length", str(len(body)))
    handler.end_headers()
    handler.wfile.write(body)


def read_json_body(handler: BaseHTTPRequestHandler) -> dict:
    length = int(handler.headers.get("Content-Length", "0") or "0")
    if length <= 0:
        return {}
    raw_body = handler.rfile.read(length)
    return json.loads(raw_body.decode("utf-8"))


### API 라우팅과 핸들러

- GET은 설정/메일 조회, POST는 다운로드/CSV 변환/단어 분석을 처리합니다.
- 각 핸들러는 전용 모듈 함수를 호출하고 결과를 JSON으로 반환합니다.

In [ ]:
class MailUiHandler(BaseHTTPRequestHandler):
    def log_message(self, format: str, *args: object) -> None:
        print(f"[mail-ui] {self.address_string()} - {format % args}")

    def do_GET(self) -> None:
        parsed = urlparse(self.path)
        if parsed.path in STATIC_FILES:
            static_response(self, STATIC_FILES[parsed.path])
            return

        if parsed.path == "/api/config":
            self.handle_config()
            return

        if parsed.path == "/api/mail":
            self.handle_mail_query(parsed.query)
            return

        self.send_error(404)

    def do_POST(self) -> None:
        parsed = urlparse(self.path)
        if parsed.path == "/api/download":
            self.handle_download()
            return

        if parsed.path == "/api/convert":
            self.handle_convert()
            return

        if parsed.path == "/api/analyze":
            self.handle_analyze()
            return

        self.send_error(404)

    def handle_config(self) -> None:
        try:
            settings = load_mail_settings()
            json_response(
                self,
                200,
                {
                    "emailAddress": settings.email_address,
                    "imapServer": settings.imap_server,
                },
            )
        except Exception as exc:
            json_response(self, 500, {"error": str(exc)})

    def handle_mail_query(self, query_string: str) -> None:
        try:
            params = parse_qs(query_string)
            search_text = params.get("query", [""])[0]
            period = params.get("period", ["7"])[0]
            mails = list_inbox_mails(search_text=search_text, period=period)
            json_response(self, 200, {"mails": mails})
        except Exception as exc:
            json_response(self, 500, {"error": str(exc)})

    def handle_download(self) -> None:
        try:
            payload = read_json_body(self)
            mail_ids = [str(mail_id) for mail_id in payload.get("mailIds", [])]
            files = download_pdf_attachments(mail_ids)
            json_response(self, 200, {"files": files})
        except Exception as exc:
            json_response(self, 500, {"error": str(exc)})

    def handle_convert(self) -> None:
        try:
            payload = read_json_body(self)
            files = [str(file_path) for file_path in payload.get("files", [])]
            result = convert_pdfs_to_csv(files)
            json_response(self, 200, result)
        except Exception as exc:
            json_response(self, 500, {"error": str(exc)})

    def handle_analyze(self) -> None:
        try:
            payload = read_json_body(self)
            files = [str(file_path) for file_path in payload.get("files", [])]
            analysis = analyze_word_counts(files)
            analysis["wordCloud"] = build_word_cloud(analysis["topWords"])
            json_response(self, 200, {"analysis": analysis})
        except Exception as exc:
            json_response(self, 500, {"error": str(exc)})


### 서버 실행 함수

- 주피터에서는 셀이 멈춰버리지 않도록 실제 실행부를 주석 처리했습니다.
- 실행은 터미널에서 `.venv\Scripts\python.exe src\mail_ui_server.py`로 합니다.

In [ ]:
# 실제 서버 실행은 터미널에서 실행합니다.
# .venv\Scripts\python.exe src\mail_ui_server.py

def main() -> None:
    server = ThreadingHTTPServer((HOST, PORT), MailUiHandler)
    print(f"mail ui server: http://{HOST}:{PORT}")
    server.serve_forever()

# 주피터 셀에서는 아래 실행부를 일부러 주석 처리합니다.
# if __name__ == "__main__":
#     main()


## web/ 프론트엔드 파일은 어디를 보면 되나

- `web/flow-ui.html`: 화면 구조와 버튼/표 영역의 id를 확인합니다.
- `web/flow-ui.js`: 버튼 클릭이 어떤 API로 이어지는지 확인합니다.
- `web/flow-ui.css`: 화면 배치와 워드클라우드 스타일을 확인합니다.
- 이 세 파일은 화면 구현 파일이라 이번 학습 섹션에서는 함수 단위로 쪼개지 않았습니다.